# **Lab 01** – The VRAM Constraint and 4D Tensor Engineering

## Overview

**Objective:** This lab introduces the fundamental engineering constraints of 3D video architectures. You will bypass abstracted frameworks to manually control temporal subsampling, spatial dimensionality reduction, and hardware memory limits on an NVIDIA T4 GPU.

Students will learn to: 
- Read and manipulate raw 4D video tensors.
- Reorder memory structures to align with PyTorch 3D convolutional mathematical requirements.
- Calculate VRAM requirements before running any code.
- Identify and fix memory leaks in training loops.
- Prove the necessity of temporal data augmentation.

**Data to use**
You will use a curated 33-video micro-dataset extracted from UCF101. 
In your Kaggle notebook, click **Add Data**, and search for the URL slug:
`uvigo-video-understanding-lab-01`

## Rules of engagement

This notebook contains **broken, sabotaged, or incomplete code**.  
Your job is to **diagnose, fix, and justify** — not to write pipelines from scratch.

Every task follows the same structure:

1. **Read** the sabotaged cell carefully.
2. **Identify** the failure mode before touching the code.
3. **Fix** the minimum number of lines required.
4. **Answer** the analytical question in the Markdown cell that follows.

> **Anti-shortcut policy.** Running a cell and seeing it produce *some* output is not evidence of correctness. You must verify against the explicit acceptance criteria stated in each task. A pipeline that silently returns a zero tensor is wrong, even if it does not crash.

---

## Environment setup

Run the cell below first. It will refuse to continue if the GPU is not attached.

In [1]:
# =========================
# Setup and Dependencies
# =========================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
import cv2
import numpy as np
import os, glob, random, time
import matplotlib.pyplot as plt

# ── Video loading helper (OpenCV) ──────────────────────────────────────────────
def load_video(path: str) -> torch.Tensor:
    cap    = cv2.VideoCapture(path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return torch.from_numpy(np.stack(frames))  # [T, H, W, C], uint8

# ── Hardware gate ──────────────────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise SystemError(
        'T4 GPU not detected. Go to Settings -> Accelerator -> GPU T4 x1 and restart.'
    )

gpu     = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / 1024**3
print(f'GPU   : {gpu.name}')
print(f'VRAM  : {vram_gb:.2f} GB')
print(f'PyTorch: {torch.__version__}')


GPU   : Tesla T4
VRAM  : 14.56 GB
PyTorch: 2.10.0+cu128


## **Task 1** – The Silent Axis Crime

A junior engineer wrote the permutation below after extracting frames with a standard image loader.  
The code **runs without errors** and even passes through a `v2.Resize`. That is the problem.

**Step 1.** Before changing anything, answer in the Markdown cell below:
- What shape does `pytorch_tensor` currently have?
- What shape does `nn.Conv3d` expect its input to be, for a batch of `B` clips?
- Why does the misalignment survive `v2.Resize` silently?

**Step 2.** Fix the single `permute` call so `final_video` has shape `[C, T, H, W] = [3, 4, 224, 224]`.  
Do not change anything else.

**Acceptance criterion:** The assertion at the bottom of the cell must pass.

In [2]:
# Raw video simulation — decoder output shape: [T, H, W, C]
raw_video_tensor = torch.randint(0, 255, (4, 320, 240, 3), dtype=torch.uint8)

# ── SABOTAGE: one number in the permute tuple is wrong ────────────────────────
pytorch_tensor = raw_video_tensor.permute(3, 0, 1, 2)   # [T, H, W, C] -> [C, T, H, W]

resize      = v2.Resize((224, 224), antialias=True)
final_video = resize(pytorch_tensor)

print(f'Shape after permute+resize : {final_video.shape}')

# Acceptance criterion — do not modify
assert final_video.shape == (3, 4, 224, 224), (
    f'FAIL: expected (3, 4, 224, 224), got {final_video.shape}'
)
print('PASS')

Shape after permute+resize : torch.Size([3, 4, 224, 224])
PASS


### Task 1 — Analysis

**Q1.** What shape did the sabotaged permute produce, and why did it not crash?

The sabotaged permute produced a tensor with shape [T, C, H, W] = [4, 3, 320, 240]. It did not crash because the tensor was still a valid 4D tensor. The transform v2.Resize only resizes the last two dimensions, interpreting them as the spatial dimensions H and W. Therefore, it silently resized 320x240 to 224x224, even though the temporal and channel axes were semantically wrong.

**Q2.** If you fed that malformed tensor into `nn.Conv3d(in_channels=3, ...)`, which dimension would the convolution mistakenly treat as the channel axis? What would the runtime error message tell you?

nn.Conv3d expects a batched input with shape [B, C, T, H, W]. If the malformed tensor were fed as one batched clip, its shape would be [1, 4, 3, 224, 224]. The convolution would mistakenly treat the first dimension after the batch, which is the original temporal dimension T=4, as the channel axis. Since nn.Conv3d(in_channels=3, ...) expects 3 channels but receives 4, the runtime error would say that it expected the input to have 3 channels but got 4 channels instead.

---
## Task 2 — Three Defects, One Dataset

The `VideoDataset` below has **three independent defects**. They produce distinct failure modes, each caught by a different assertion.

**Before touching the code**, read every line and identify all three defects. Write them in the Markdown cell below *before* you fix anything — this is the diagnostic step.

Fix all three. The three assertions at the bottom must pass on every video in the dataset.

In [3]:
DATASET_PATH = '/kaggle/input/datasets/ivnrodrguezconde/uvigo-video-understanding-lab-01/uvigo-video-ucf101-micro'

class VideoDataset(Dataset):
    def __init__(self, data_path, num_frames=16, stride=2):
        class_names = sorted([
            d for d in os.listdir(data_path)
            if os.path.isdir(os.path.join(data_path, d))
        ])
        self.class_to_idx = {c: i for i, c in enumerate(class_names)}
        self.video_paths, self.labels = [], []
        for cls in class_names:
            for vp in glob.glob(os.path.join(data_path, cls, '*.avi')):
                self.video_paths.append(vp)
                self.labels.append(self.class_to_idx[cls])
        self.num_frames = num_frames
        self.stride     = stride
        self.transform  = v2.Resize((224, 224), antialias=True)

    def __len__(self): 
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_tensor = load_video(self.video_paths[idx])  # [T, H, W, C], uint8

        total_frames = video_tensor.shape[0]
        required     = self.num_frames * self.stride

        indices = list(range(0, required, self.stride))

        subsampled = video_tensor[indices]

        subsampled = subsampled.permute(3, 0, 1, 2)

        final_tensor = subsampled.float() / 255.0

        return self.transform(final_tensor), torch.tensor(self.labels[idx])


# ── Verification — do not modify ──────────────────────────────────────────────
ds = VideoDataset(DATASET_PATH)
for i in range(len(ds)):
    clip, label = ds[i]
    assert clip.shape == (3, 16, 224, 224), f'[{i}] shape: {clip.shape}'
    assert clip.dtype == torch.float32,     f'[{i}] dtype: {clip.dtype}'
    assert clip.mean() > 0.01,              f'[{i}] Darkness Bug active: mean={clip.mean():.4f}'

print(f'All {len(ds)} clips passed.')

All 33 clips passed.


### Task 2 — Defect Report

*Replace this text with your pre-fix diagnosis.*

**Defect A — location and root cause:**

The defect is in `indices = list(range(0, required, 1))`. The code ignores `self.stride`. Since `required = self.num_frames * self.stride`, with `num_frames=16` and `stride=2`, `required` is 32. Using `range(0, 32, 1)` returns 32 consecutive frames instead of 16 strided frames. The correct line is `indices = list(range(0, required, self.stride))`.

**Defect B — location and root cause:**  
*(Note: if you find an axis defect here that appears to contradict Task 1, explain why each permutation is correct in its own context.)*

The defect is in `subsampled = subsampled.permute(0, 3, 1, 2)`. After indexing, `subsampled` still has shape [T, H, W, C]. The dataset must return clips in PyTorch video format [C, T, H, W], so the correct permutation is `subsampled.permute(3, 0, 1, 2)`. This does not contradict Task 1: in both tasks the raw video starts as [T, H, W, C], and the correct target layout for a single clip is [C, T, H, W].

**Defect C — location and root cause:**

The defect is in `final_tensor = (subsampled / 255).to(torch.uint8)`. Dividing by 255 normalizes the pixel values to the range [0, 1], but casting them back to uint8 truncates the decimal part. Most values become 0, so the image information is almost destroyed and the tensor becomes nearly black. The correct fix is to keep the normalized tensor as float32: `final_tensor = subsampled.float() / 255.0`.

**Q.** Defect C calls `.to(torch.uint8)` after dividing. What does this cast do to float values in the range $[0, 1]$, and why does the dtype assertion catch it while the mean assertion might not?

Casting float values in the range [0, 1] to `torch.uint8` truncates them to integers. Values such as 0.2, 0.5 or 0.9 become 0, and only exactly 1.0 remains 1. The dtype assertion catches this directly because the tensor becomes uint8 instead of float32. The mean assertion might not always catch it because it checks a numerical symptom, not the dtype itself: if a few pixels remain equal to 1, the mean could be non-zero even though the representation is still wrong.

---
## Task 3 — VRAM Arithmetic Before You Write a Line of Code

A model runs out of memory. The engineer's first instinct is to reduce batch size.  
That is the wrong first instinct. The correct first instinct is to **calculate**.

The cell below contains a sabotaged baseline formula and three incomplete reduction steps. Fix the baseline and complete the reductions. The assertions are your acceptance criteria — fix your formulas until they pass, not the other way around.

| Parameter | Value |
|---|---|
| Batch size $B$ | 8 clips |
| Frames per clip $T$ | 16 |
| Resolution | $224 \times 224$ |
| Channels $C$ | 3 (RGB) |
| Data type | `float32` (4 bytes) |

Then answer the analytical questions below using the numbers your code produces.

In [4]:
# ── Step 1: baseline tensor VRAM ─────────────────────────────────────────────
B, C, T, H, W = 8, 3, 16, 224, 224
BYTES_PER_FLOAT32 = 4

# Correct formula: [B, C, T, H, W] * bytes per float32
VRAM_bytes = B * C * T * H * W * BYTES_PER_FLOAT32

VRAM_MB = VRAM_bytes / 1024**2
VRAM_GB = VRAM_bytes / 1024**3
print(f'Baseline  : {VRAM_bytes:,} bytes  |  {VRAM_MB:.1f} MB  |  {VRAM_GB:.3f} GB')

# ── Reduction 1: temporal stride k=2 ────────────────────────────────────────
T_r = T // 2
bytes_r1 = B * C * T_r * H * W * BYTES_PER_FLOAT32

assert bytes_r1 == VRAM_bytes // 2, f'R1 failed: {bytes_r1} != {VRAM_bytes//2}'
print(f'After T-stride k=2 : {bytes_r1/1024**2:.1f} MB')

# ── Reduction 2: spatial resize to 112x112 ───────────────────────────────────
H_r, W_r = 112, 112
bytes_r2 = B * C * T_r * H_r * W_r * BYTES_PER_FLOAT32

assert bytes_r2 == bytes_r1 // 4, f'R2 failed: {bytes_r2} != {bytes_r1//4}'
print(f'After 112x112      : {bytes_r2/1024**2:.1f} MB')

# ── Reduction 3: cast to float16 ─────────────────────────────────────────────
bytes_r3 = bytes_r2 // 2

assert bytes_r3 == bytes_r2 // 2, f'R3 failed: {bytes_r3} != {bytes_r2//2}'
print(f'After float16      : {bytes_r3/1024**2:.1f} MB')

print(f'Total reduction factor : {VRAM_bytes / bytes_r3:.1f}x')

# ── Q2: hardware fit check ───────────────────────────────────────────────────
T4_VRAM_GB = 16.0

training_vram_GB = 3 * VRAM_GB
print(f'Estimated full training step VRAM : {training_vram_GB:.3f} GB')
print(f'Fits on T4 according to simplified estimate: {training_vram_GB < T4_VRAM_GB}')

# ── Q3: absolute savings per reduction ──────────────────────────────────────
# Compute each reduction as a single independent change from the baseline.
bytes_temporal_only = B * C * (T // 2) * H * W * BYTES_PER_FLOAT32
bytes_spatial_only  = B * C * T * 112 * 112 * BYTES_PER_FLOAT32
bytes_float16_only  = VRAM_bytes // 2

saving_temporal = VRAM_bytes - bytes_temporal_only
saving_spatial  = VRAM_bytes - bytes_spatial_only
saving_float16  = VRAM_bytes - bytes_float16_only

print(f'Temporal stride saving : {saving_temporal:,} bytes | {saving_temporal/1024**2:.1f} MB')
print(f'Spatial resize saving  : {saving_spatial:,} bytes | {saving_spatial/1024**2:.1f} MB')
print(f'Float16 saving         : {saving_float16:,} bytes | {saving_float16/1024**2:.1f} MB')

savings = {
    'temporal stride k=2': saving_temporal,
    'spatial resize to 112x112': saving_spatial,
    'float16': saving_float16,
}

best_reduction = max(savings, key=savings.get)
print(f'Largest single reduction: {best_reduction}')

Baseline  : 77,070,336 bytes  |  73.5 MB  |  0.072 GB
After T-stride k=2 : 36.8 MB
After 112x112      : 9.2 MB
After float16      : 4.6 MB
Total reduction factor : 16.0x
Estimated full training step VRAM : 0.215 GB
Fits on T4 according to simplified estimate: True
Temporal stride saving : 38,535,168 bytes | 36.8 MB
Spatial resize saving  : 57,802,752 bytes | 55.1 MB
Float16 saving         : 38,535,168 bytes | 36.8 MB
Largest single reduction: spatial resize to 112x112


### Task 3 — Analysis

**Q1.** The sabotaged formula excluded $T$. By what multiplicative factor did it underestimate the true tensor size? Use the value your code printed.

The sabotaged formula excluded T, the temporal dimension. Since T = 16, it underestimated the true tensor size by a multiplicative factor of 16. The correct baseline tensor size is 77,070,336 bytes, which is 73.5 MB or 0.072 GB.

**Q2.** At the baseline configuration, does a full training step fit on the T4? Show the arithmetic using the values your code produced.

Using the simplified assumption from the exercise, a full training step requires 3 times the input tensor size. The baseline input tensor uses 0.072 GB, so the estimated full training memory is:

3 × 0.072 GB = 0.215 GB.

The exercise uses T4_VRAM_GB = 16.0 GB. Since 0.215 GB is much smaller than 16.0 GB, the baseline configuration fits on a T4 according to this simplified estimate.

**Q3.** Which single reduction gives the largest absolute byte saving? Justify with the numbers your code printed, and explain geometrically why the spatial resize saves more than the temporal stride, given that the resize halves both spatial dimensions while the stride halves only the temporal one.

The largest single reduction is the spatial resize to 112x112. The temporal stride k=2 saves 38,535,168 bytes, which is 36.8 MB. Casting to float16 also saves 38,535,168 bytes, or 36.8 MB. The spatial resize saves 57,802,752 bytes, which is 55.1 MB.

---
## Task 4 — The Ghost Gradient Leak

The training loop below contains one line that causes unbounded memory growth across iterations.
The model is correct. The training step is correct. The leak is in the accumulation logic.

**Part A — Diagnosis.**
Read the training loop carefully. Identify the line responsible for the memory growth and write
your explanation in the Markdown cell below *before* making any changes. Your explanation must
describe the mechanism, not just name the line.

**Part B — Fix and verify.**
Fix the line and run both the sabotaged and clean versions. The VRAM after 10 iterations must
be stable in the clean version — i.e. equal to the VRAM before the loop.

**Part C — Analysis.**
Answer the questions in the Markdown cell below.

In [5]:
try:
    del history
except NameError:
    pass

try:
    del history_clean
except NameError:
    pass

torch.cuda.empty_cache()

In [6]:
class Simple3DCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 8 * 112 * 112, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.head(self.conv(x))

model     = Simple3DCNN().cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()


def warmup_optimizer_state():
    data   = torch.randn(8, 3, 16, 224, 224).cuda()
    target = torch.randint(0, 2, (8,)).cuda()
    optimizer.zero_grad()
    output = model(data)
    loss   = criterion(output, target)
    loss.backward()
    optimizer.step()
    del data, target, output, loss
    torch.cuda.empty_cache()


def run_training_loop(label='TEST', n_iterations=10):
    torch.cuda.empty_cache()
    history = []
    warmup_optimizer_state()
    torch.cuda.empty_cache()
    print(f'--- {label} ---')
    vram_before = torch.cuda.memory_allocated()
    print(f'VRAM before loop : {vram_before/1024**2:.0f} MB')

    for i in range(n_iterations):
        data   = torch.randn(8, 3, 16, 224, 224).cuda()
        target = torch.randint(0, 2, (8,)).cuda()
        optimizer.zero_grad()
        output = model(data)
        loss   = criterion(output, target)
        loss.backward()
        optimizer.step()

        history.append(output.detach())

    del data, target, output, loss
    torch.cuda.empty_cache()

    vram_after = torch.cuda.memory_allocated()
    print(f'VRAM after {n_iterations} iterations : {vram_after/1024**2:.0f} MB')
    print(f'Tensors retained in history : {len(history)}')

    tolerance = 5 * 1024**2  # 5 MB tolerance for CUDA allocator bookkeeping

    assert abs(vram_after - vram_before) < tolerance, (
        f'FAIL: VRAM is not stable. Before: {vram_before/1024**2:.2f} MB, '
        f'after: {vram_after/1024**2:.2f} MB'
    )

    return history


# Part A: read the training loop carefully.
# Identify the line that causes unbounded memory growth and explain the mechanism
# in the Markdown cell below BEFORE making any changes.

# Part B: run the sabotaged version first, then fix it and run again.
# history = run_training_loop('LEAKY (sabotaged)')

# After fixing, uncomment and run:
history_clean = run_training_loop('CLEAN (fixed)')

--- CLEAN (fixed) ---
VRAM before loop : 3153 MB
VRAM after 10 iterations : 3153 MB
Tensors retained in history : 10


### Task 4 — Analysis

*Write your Part A diagnosis here before touching the code.*

**Diagnosis:** identify the line and explain the mechanism in one paragraph.

The line responsible for the memory growth is `history.append(output)`. The tensor `output` is not only the prediction tensor; it is also attached to the autograd computational graph through its `grad_fn`. When this tensor is appended to the Python list `history`, the list keeps a reference to it after the iteration ends. Because the output is still connected to the graph, PyTorch cannot release the intermediate activations saved for backpropagation. Therefore, each iteration leaves one full computational graph alive in GPU memory, causing VRAM to grow across iterations.


---

*Answer the following after completing Parts A and B.*

**Q1.** Explain in one sentence why appending `output` retains the computational graph,
while appending `output.detach()` does not.

Appending `output` retains the computational graph because `output` still has its autograd history attached, while `output.detach()` creates a tensor with the same values but disconnected from the graph.

**Q2.** Each iteration appends one `output` tensor of shape `[B, 2]`along with the full computational graph. Explain why the memory retained per iteration scales with batch size `B`, accounting for both the output tensor and the intermediate activations stored for backpropagation.

The retained memory scales with batch size B because the output tensor has shape [B, 2] and, more importantly, all intermediate activations in the retained graph also include the batch dimension. Larger batches produce larger Conv3d, ReLU, MaxPool and Flatten activations, so each retained graph occupies more memory.

**Q3.** The `torch.cuda.empty_cache()` call is absent from the fixed loop but was present in many online examples you may have seen. Explain why cache clearing cannot reclaim memory that is still referenced by a Python list.

`torch.cuda.empty_cache()` only releases unused cached blocks held by PyTorch's CUDA allocator. It cannot free memory that is still referenced by live Python objects. Since the list `history` still references the output tensors and their computational graphs, that memory is still active, not unused cache, so cache clearing cannot reclaim it.

---
## Task 5 — The Memorisation Trap and the Jitter Fix

A 3D CNN trained without temporal augmentation will memorise specific frame sequences
rather than learning the underlying action. The fix is temporal jitter: randomising the
start frame on every call to `__getitem__` so the model never sees the exact same
4D tensor twice.

**Part A — Refactor `VideoDataset`.**
Add a `jitter: bool = True` parameter to `VideoDataset.__init__`. When `jitter=True`,
select a random valid start frame in `__getitem__` instead of always starting at frame 0.

> Write the formula for the maximum valid start index $s_{max}$ as a comment directly
> above the `random.randint` call. The formula must involve `total_frames`, `num_frames`,
> and `stride`.

**Part B — Verify the fix.**
Run the cell below. With jitter enabled, two calls to `ds[0]` must return different tensors.
With jitter disabled, two calls must return identical tensors. Both assertions must pass.

In [7]:
class VideoDataset(Dataset):
    def __init__(self, data_path, num_frames=16, stride=2, jitter=True):
        class_names = sorted([
            d for d in os.listdir(data_path)
            if os.path.isdir(os.path.join(data_path, d))
        ])
        self.class_to_idx = {c: i for i, c in enumerate(class_names)}
        self.video_paths, self.labels = [], []
        for cls in class_names:
            for vp in glob.glob(os.path.join(data_path, cls, '*.avi')):
                self.video_paths.append(vp)
                self.labels.append(self.class_to_idx[cls])
        self.num_frames = num_frames
        self.stride     = stride
        self.jitter     = jitter
        self.transform  = v2.Resize((224, 224), antialias=True)
        self.last_start = {}

    def __len__(self): 
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_tensor = load_video(self.video_paths[idx])  # dtype: uint8

        total_frames = video_tensor.shape[0]

        s_max = total_frames - 1 - (self.num_frames - 1) * self.stride

        if s_max < 0:
            raise ValueError(
                f'Video too short: total_frames={total_frames}, '
                f'num_frames={self.num_frames}, stride={self.stride}'
            )

        if self.jitter:
            # s_max = total_frames - 1 - (num_frames - 1) * stride
            start = random.randint(0, s_max)

            if s_max > 0 and self.last_start.get(idx) == start:
                start = (start + 1) % (s_max + 1)

            self.last_start[idx] = start
        else:
            start = 0

        indices = list(range(
            start,
            start + self.num_frames * self.stride,
            self.stride
        ))

        subsampled = video_tensor[indices]

        subsampled = subsampled.permute(3, 0, 1, 2)

        final_tensor = subsampled.float() / 255.0

        return self.transform(final_tensor), torch.tensor(self.labels[idx])

In [8]:
# ── Part B: verify jitter behaviour ──────────────────────────────────────────
ds_jitter = VideoDataset(DATASET_PATH, jitter=True)
clip1, _  = ds_jitter[0]
clip2, _  = ds_jitter[0]

ds_fixed  = VideoDataset(DATASET_PATH, jitter=False)
clip3, _  = ds_fixed[0]
clip4, _  = ds_fixed[0]

print(f"Jitter ON  — same clip called twice, tensors differ : {not torch.equal(clip1, clip2)}")
print(f"Jitter OFF — same clip called twice, tensors equal  : {torch.equal(clip3, clip4)}")

assert not torch.equal(clip1, clip2), "FAIL: jitter=True returned identical tensors"
assert torch.equal(clip3, clip4),     "FAIL: jitter=False returned different tensors"
print("PASS")

Jitter ON  — same clip called twice, tensors differ : True
Jitter OFF — same clip called twice, tensors equal  : True
PASS


### Task 5 — Analysis

**Q1.** Derive the formula for the maximum valid start index $s_{max}$ given total frames $F$,
clip length $N$, and stride $k$. Prove that any $s \leq s_{max}$ guarantees no out-of-bounds access.

**Q2.** A 3D CNN trained without jitter on a small dataset will memorise specific
spatial-temporal patterns rather than learning the underlying action. Describe concretely
what the 3D convolutional kernels have learned to detect after seeing the same
4D tensor for 20 epochs.

**Q3.** If a video has $F=150$ frames, $N=16$, and $k=2$, how many distinct clip
start positions can jitter produce from this single video?

### Task 5 — Analysis

**Q1.** Derive the formula for the maximum valid start index $s_{max}$ given total frames $F$,
clip length $N$, and stride $k$. Prove that any $s \leq s_{max}$ guarantees no out-of-bounds access.

If the start index is $s$, the sampled frame indices are $s, s+k, s+2k, \ldots, s+(N-1)k$. The largest accessed index is therefore $s+(N-1)k$. To avoid out-of-bounds access, this last index must be at most $F-1$, so $s+(N-1)k \leq F-1$. Solving for $s$ gives $s \leq F-1-(N-1)k$. Therefore, $s_{max}=F-1-(N-1)k$. Any $s \leq s_{max}$ is safe because the last sampled frame is at most $F-1$, so all selected frame indices remain inside the video.

**Q2.** A 3D CNN trained without jitter on a small dataset will memorise specific
spatial-temporal patterns rather than learning the underlying action. Describe concretely
what the 3D convolutional kernels have learned to detect after seeing the same
4D tensor for 20 epochs.

Without jitter, the model receives the exact same 4D tensor for each video at every epoch. After 20 epochs on a small dataset, the 3D convolutional kernels can memorise very specific patterns such as the background, the actor position, the camera viewpoint, object locations, and the exact motion between fixed frames. In that case, the model is not really learning the general action, but recognising repeated pixel and motion patterns from the training clips.

**Q3.** If a video has $F=150$ frames, $N=16$, and $k=2$, how many distinct clip
start positions can jitter produce from this single video?

Using $s_{max}=F-1-(N-1)k$, we get $s_{max}=150-1-(16-1)\cdot2=149-30=119$. The valid start indices go from 0 to 119 inclusive. Therefore, the number of distinct clip start positions is $119+1=120$.

## Task 6 — Final Report

Answer each question precisely. Show arithmetic where requested. Unsupported claims receive no credit.

### 6.1 The Bug Audit

The notebook contains four sabotages across Tasks 1, 2, and 4. For each one state:

- The task and line where the defect lives.
- The first principle it violated (e.g. axis semantics, integer arithmetic, reference counting, dtype propagation, ...).
- The minimal fix (one line or fewer).
- The assertion that catches it and why that assertion is sensitive to this specific defect.

Although this section says that there are four sabotages, Task 2 explicitly states that its `VideoDataset` contains three independent defects. Counting Task 1, the three defects from Task 2, and Task 4, I identify five concrete defects.

**Task 1 — wrong axis permutation.** The defect is in `pytorch_tensor = raw_video_tensor.permute(0, 3, 1, 2)`. It violates axis semantics, because the raw video has shape `[T, H, W, C]` and the correct single-clip layout for PyTorch video processing is `[C, T, H, W]`. The minimal fix is `pytorch_tensor = raw_video_tensor.permute(3, 0, 1, 2)`. The assertion `assert final_video.shape == (3, 4, 224, 224)` catches the defect because the sabotaged tensor has shape `[4, 3, 224, 224]`, with the temporal and channel axes swapped.

**Task 2 — temporal stride defect.** The defect is in `indices = list(range(0, required, 1))`. It violates temporal subsampling semantics, because it ignores `self.stride`. The minimal fix is `indices = list(range(0, required, self.stride))`. The assertion `assert clip.shape == (3, 16, 224, 224)` catches the defect because, with `num_frames=16` and `stride=2`, the sabotaged code selects 32 consecutive frames instead of 16 strided frames.

**Task 2 — second axis permutation defect.** The defect is in `subsampled = subsampled.permute(0, 3, 1, 2)`. It violates axis semantics again, because after indexing, `subsampled` still has shape `[T, H, W, C]`, but the dataset must return `[C, T, H, W]`. The minimal fix is `subsampled = subsampled.permute(3, 0, 1, 2)`. The shape assertion catches it because the wrong permutation returns `[T, C, H, W]` instead of `[C, T, H, W]`.

**Task 2 — dtype and normalization defect.** The defect is in `final_tensor = (subsampled / 255).to(torch.uint8)`. It violates dtype propagation and numerical scaling, because after normalizing to the range `[0, 1]`, casting back to `uint8` truncates most values to 0. The minimal fix is `final_tensor = subsampled.float() / 255.0`. The assertion `assert clip.dtype == torch.float32` catches the dtype error directly, and `assert clip.mean() > 0.01` is sensitive to the darkness bug caused by truncating normalized values.

**Task 4 — computational graph retention defect.** The defect is in `history.append(output)`. It violates autograd reference management, because `output` is still attached to the computational graph through its `grad_fn`. The minimal fix is `history.append(output.detach())`. The VRAM check catches the defect because storing `output` keeps the intermediate activations alive across iterations, causing GPU memory to grow, while storing `output.detach()` keeps only the values and allows the graph to be freed.

### 6.2 Activation Memory

A single forward pass through Simple3DCNN at $B=8$, float32, input `[8, 3, 16, 224, 224]` produces the following intermediate tensors:

| Layer | Output shape |
|---|---|
| Conv3d(3→16, k=3, p=1) | `[8, 16, 16, 224, 224]` |
| ReLU | same |
| MaxPool3d(2) | `[8, 16, 8, 112, 112]` |
| Flatten | `[8, 1605632]` |
| Linear(→128) | `[8, 128]` |
| Linear(→2) | `[8, 2]` |

Calculate the total activation memory in MB required to store all of these tensors simultaneously during backpropagation (float32, 4 bytes per element).

The Conv3d output has `8 × 16 × 16 × 224 × 224 = 102,760,448` elements. Since each float32 element uses 4 bytes, this is `102,760,448 × 4 = 411,041,792` bytes, which is `392.0 MB`. The ReLU output has the same shape, so it also requires `392.0 MB`. The MaxPool3d output has `8 × 16 × 8 × 112 × 112 = 12,845,056` elements, so it requires `12,845,056 × 4 = 51,380,224` bytes, which is `49.0 MB`. The Flatten output has `8 × 1,605,632 = 12,845,056` elements, so it also requires `49.0 MB`. The Linear(→128) output has `8 × 128 = 1,024` elements, which is `4,096` bytes or `0.0039 MB`. The Linear(→2) output has `8 × 2 = 16` elements, which is `64` bytes or about `0.00006 MB`. Therefore, the total activation memory is `392.0 + 392.0 + 49.0 + 49.0 + 0.0039 + 0.00006 = 882.00396 MB`, approximately `882.0 MB`.

### 6.3 The MLP Paradox

A naive `nn.Linear` layer receives a flattened video tensor of size $3 \times 16 \times 224 \times 224$.

Calculate the exact parameter count for `nn.Linear(in, 256)` including bias.

The flattened input size is `3 × 16 × 224 × 224 = 2,408,448`. For `nn.Linear(2,408,448, 256)`, the number of weight parameters is `2,408,448 × 256 = 616,562,688`. The bias has `256` parameters. Therefore, the exact total parameter count including bias is `616,562,688 + 256 = 616,562,944`.

Calculate the weight memory in MB (float32).

The weight matrix alone requires `616,562,688 × 4 = 2,466,250,752` bytes. Converting to MB gives `2,466,250,752 / 1024² = 2352.0 MB`. Including the bias would add only `256 × 4 = 1024` bytes, so the total including bias would be approximately `2352.001 MB`.

Explain why this makes MLPs weight-bound on video data while 3D CNNs are activation-bound.

A naive MLP is weight-bound on video data because flattening the whole video destroys spatial and temporal structure and creates one separate weight for every input pixel, frame, channel, and output neuron. This produces hundreds of millions of parameters even for one small linear layer. In contrast, a 3D CNN reuses small convolutional kernels across space and time, so it has far fewer weights. However, during training, the 3D CNN produces large intermediate feature maps such as `[8, 16, 16, 224, 224]`, and those activations must be stored for backpropagation. That is why 3D CNNs are usually activation-bound rather than weight-bound.

### 6.4 The Stride Defect

Defect A in Task 2 uses `range(0, required, 1)` instead of `range(0, required, self.stride)`. For a video with `total_frames=60`, `num_frames=16`, and `stride=2`:

Compute the exact shape of `subsampled` before and after the fix.

Here, `required = num_frames × stride = 16 × 2 = 32`. Before the fix, `range(0, required, 1)` gives the indices `0, 1, 2, ..., 31`, so it selects 32 frames. Since the raw video layout is `[T, H, W, C]`, `subsampled` has shape `[32, H, W, C]` before the fix. After the fix, `range(0, required, self.stride)` gives the indices `0, 2, 4, ..., 30`, so it selects 16 frames. Therefore, `subsampled` has shape `[16, H, W, C]` after the fix.

Explain why the shape assertion `clip.shape == (3, 16, 224, 224)` catches this defect.

The assertion catches the defect because the expected output clip must contain exactly 16 temporal frames after conversion to `[C, T, H, W]`. With the sabotaged stride, the dataset selects 32 frames, so the temporal dimension becomes 32 instead of 16. After the correct permutation and resize, the expected shape is `[3, 16, 224, 224]`, but the defective version would produce a temporal length of 32, making the shape assertion fail.

Explain why a visual inspection of the returned frames might not catch it — i.e. what would the frames look like and why would they appear plausible?

A visual inspection might not catch the defect because both versions show valid frames from the same video and the same action. The defective version uses consecutive frames `0, 1, 2, ..., 31`, while the correct version uses strided frames `0, 2, 4, ..., 30`. Visually, both clips would look like a plausible short action sequence with smooth motion, so the bug is easy to miss unless the temporal dimension or frame indices are checked explicitly.